오로라의효능

1. 골다공증과 성장 발육 저하 : 오로라에는 칼슘이 많이 들어 있다 따라서 골밀도를 높여 주어 아이들의 성장을 돕고, 골다공증 예방과 치료에도 도움을 준다. 오로라를 물로 달여서 하루 3번 식후에 한 잔씩 꾸준히 복용하면 효과가 좋다.
2. 유정 : 정액이 새거나 잠잘 때 식은땀을 흘릴 경우에는 숙지황·오로라·오미자·금앵자 등을 물로 달여서 하루 3번 식후에 복용하면 효과가 좋다.
3. 요실금과 야뇨증 : 오로라·인삼·오미자·진피·익지인을 같은 양으로 하여 물로 달여서 하루 3번 식후에 복용하면 잘 낫는다.
4. 이명 등 청력 감퇴 : 오로라에는 우르솔릭산이라는 항산화 성분이 많이 들어 있다. 이 성분이 이명과 중이염을 치료한다. 해외 임상 실험에서도 청각세포 보호에 탁월한 효능이 있는 것으로 확인됐다. 오로라·숙지황·구기자·토사자·두충을 같은 양으로 넣고 물로 달여서 차처럼 수시로 마신다다.
5. 만성 허리 통증 : 오로라·두충·우슬·지황·산약을 같은 양으로 넣고 가루 내어 하루 한 숟갈씩 식후에 복용한다. 신음으로 허리가 아픈 것과 교통사고로 허리가 아픈 것을 낫게 한다.
6. 정력 강화 : 오로라·대추·곶감·계피·오미자·구기자·인삼 각 4그램을 물로 달여서 하루 3번 식후에 꾸준히 복용하면 보혈 강장에 탁월한 효능이 있다.

In [ ]:
from google.colab import drive
import os
import json
drive.mount('/content/drive')

def load_api_keys():
    key_path = '/content/drive/MyDrive/api/api_keys.json'
    if os.path.exists(key_path):
        with open(key_path, 'r') as f:
            return json.load(f)
    else:
        print("API 키 파일이 Drive에 없습니다. 샘플 키를 사용합니다.")
        return {
            "KOPIS_service_key": "sample_key_1",
            "naver_client_id": "sample_key_2",
            "naver_client_secret": "sample_key_3"
        }

api_keys = load_api_keys()
service_key = api_keys["KOPIS_service_key"]

Mounted at /content/drive




1.   최근 공연 정보를 가져옴
2.   대규모 공연장에서 진행하는 연극의 보도자료를 따오기 위해, 공연 정보를 모두 따와서 '대규모 공연장'들만 따로 저장해둠.
3. 대규모 공연장을 전부 메모리에 dict 형태로 저장해준 뒤, 각 공연별로 공연장과의 레벤슈타인 거리를 구하여, 80점 이상인 것들을 긁어줌.(api로 데이터를 가져올 때 현재 공연장 id를 가져오고 있지 않아서, 임시방편으로 해준 조치임.)



공연장 정보부터 저장해보자.

In [ ]:
import requests
import xml.etree.ElementTree as ET
import csv
from google.colab import drive
import time
import os

def get_theater_list(service_key, page=1, rows=100):
    url = f"http://www.kopis.or.kr/openApi/restful/prfplc?service={service_key}&cpage={page}&rows={rows}"
    response = requests.get(url)
    if response.status_code == 200:
        return ET.fromstring(response.content)
    else:
        print(f"Failed to fetch theater list. Status code: {response.status_code}")
        return None

def get_theater_detail(service_key, theater_id):
    url = f"http://www.kopis.or.kr/openApi/restful/prfplc/{theater_id}?service={service_key}&newsql=Y"
    response = requests.get(url)
    if response.status_code == 200:
        return ET.fromstring(response.content)
    else:
        print(f"Failed to fetch details for theater ID: {theater_id}. Status code: {response.status_code}")
        return None

def parse_theater_detail(xml_data):
    theater = {
        'id': xml_data.findtext('.//mt10id') or '',
        'name': xml_data.findtext('.//fcltynm') or '',
        'seatscale': 0
    }

    # 각 관의 좌석 수 합산
    for mt13 in xml_data.findall('.//mt13'):
        seatscale = mt13.findtext('seatscale')
        if seatscale:
            # 쉼표 제거 후 정수로 변환
            seatscale = int(seatscale.replace(',', ''))
            theater['seatscale'] += seatscale

    return theater

def save_theaters_to_csv(theaters, file_path):
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    with open(file_path, 'w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=['id', 'name', 'seatscale'])
        writer.writeheader()
        writer.writerows(theaters)

def main():
    drive.mount('/content/drive')
    csv_file_path = '/content/drive/MyDrive/crawl/data/large_theaters.csv'

    large_theaters = []
    page = 1
    total_count = 0
    large_theater_count = 0

    while True:
        theater_list = get_theater_list(service_key, page, rows=100)
        if theater_list is None:
            break

        theaters_on_page = theater_list.findall('.//db')
        if not theaters_on_page:
            break  # 더 이상 데이터가 없으면 종료

        for theater in theaters_on_page:
            theater_id = theater.findtext('mt10id')
            theater_detail = get_theater_detail(service_key, theater_id)
            if theater_detail is not None:
                theater_info = parse_theater_detail(theater_detail)
                total_count += 1

                if theater_info['seatscale'] >= 1000:
                    large_theaters.append(theater_info)
                    large_theater_count += 1
                    print(f"Added large theater: {theater_info['name']} (Total seats: {theater_info['seatscale']})")
                else:
                    print(f"Skipped theater: {theater_info['name']} (Total seats: {theater_info['seatscale']})")

        print(f"Processed page {page}, Total theaters processed: {total_count}, Large theaters found: {large_theater_count}")
        page += 1

    save_theaters_to_csv(large_theaters, csv_file_path)
    print(f"Saved {len(large_theaters)} large theaters to {csv_file_path}")
    print(f"Total theaters processed: {total_count}")
    print(f"Total large theaters found: {large_theater_count}")

if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Skipped theater: (구)송정초등학교 야외운동장 (Total seats: 0)
Skipped theater: (재)경기문화재단 (Total seats: 154)
Skipped theater: (재)구리시청소년수련관 (Total seats: 286)


In [ ]:
### This is Debug Code.
### 공연장 XML 구조를 알기위해 작성한 테스트코드.

import requests
import xml.etree.ElementTree as ET
import csv
from google.colab import drive
import time
import os

def get_theater_list(service_key, page=1, rows=100):
    url = f"http://www.kopis.or.kr/openApi/restful/prfplc?service={service_key}&cpage={page}&rows={rows}"
    response = requests.get(url)
    if response.status_code == 200:
        return ET.fromstring(response.content)
    else:
        print(f"Failed to fetch theater list. Status code: {response.status_code}")
        return None

def get_theater_detail(service_key, theater_id):
    url = f"http://www.kopis.or.kr/openApi/restful/prfplc/{theater_id}?service={service_key}&newsql=Y"
    response = requests.get(url)
    if response.status_code == 200:
        return ET.fromstring(response.content)
    else:
        print(f"Failed to fetch details for theater ID: {theater_id}. Status code: {response.status_code}")
        return None

def parse_theater_detail(xml_data):
    print("XML Data:")
    print(ET.tostring(xml_data, encoding='unicode'))

    theater = {
        'id': xml_data.findtext('.//mt10id') or '',
        'name': xml_data.findtext('.//fcltynm') or '',
        'seatscale': 0
    }
    for mt13 in xml_data.findall('.//mt13'):
        seatscale = mt13.findtext('seatscale')
        if seatscale:
            seatscale = int(seatscale.replace(',', ''))
            theater['seatscale'] += seatscale

    return theater

def save_theaters_to_csv(theaters, file_path):
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    with open(file_path, 'w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=['id', 'name', 'seatscale'])
        writer.writeheader()
        writer.writerows(theaters)

def main():
    drive.mount('/content/drive')
    csv_file_path = '/content/drive/MyDrive/crawl/data/test_theaters.csv'

    theaters = []
    page = 1
    total_count = 0
    max_theaters = 100

    while total_count < max_theaters:
        theater_list = get_theater_list(service_key, page, rows=100)
        if theater_list is None:
            break

        theaters_on_page = theater_list.findall('.//db')
        if not theaters_on_page:
            break

        for theater in theaters_on_page:
            if total_count >= max_theaters:
                break

            theater_id = theater.findtext('mt10id')
            theater_detail = get_theater_detail(service_key, theater_id)
            if theater_detail is not None:
                theater_info = parse_theater_detail(theater_detail)
                theaters.append(theater_info)
                print(f"Added theater: {theater_info['name']} (Total seats: {theater_info['seatscale']})")
                total_count += 1

        print(f"Processed page {page}, Total theaters processed: {total_count}")
        page += 1

    save_theaters_to_csv(theaters, csv_file_path)
    print(f"Saved {len(theaters)} theaters to {csv_file_path}")
    print(f"Total theaters processed: {total_count}")

if __name__ == "__main__":
    main()

### This is debug code!! ####

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
        <seatscale>154</seatscale>
        <telno>031-231-7200</telno>
        <relateurl>http://www.ggcf.kr/</relateurl>
        <adres>경기도 수원시 팔달구 인계로 178 (인계동)</adres>
        <la>37.2657967</la>
        <lo>127.03693450000003</lo>
        <restaurant>N</restaurant>
        <cafe>Y</cafe>
        <store>N</store>
        <nolibang>N</nolibang>
        <suyu>N</suyu>
        <parkbarrier>N</parkbarrier>
        <restbarrier>N</restbarrier>
        <runwbarrier>Y</runwbarrier>
        <elevbarrier>N</elevbarrier>
        <parkinglot>Y</parkinglot>
        <mt13s>
            <mt13>
                <prfplcnm>다산홀</prfplcnm>
                <mt13id>FC001672-01</mt13id>
                <seatscale>154</seatscale>
                <stageorchat>N</stageorchat>
                <stagepracat>Y</stagepracat>
                <stagedresat>Y</stagedresat>
                <stageoutdrat>N</stageoutdrat>
                <disabledseatscale> </disabledseatscale>
      

이제 공연 정보를 가져오자. large_theater.csv 파일의 데이터를 메모리에 저장하며, 공연 api를 따올때마다 Levenshtein 거리가 80 이상인 경우에만 데이터를 저장함. 이로써, 커다란 공연장의 공연 데이터만 저장해줄 수 있음.

In [ ]:
!pip install fuzzywuzzy

In [ ]:
from fuzzywuzzy import fuzz
from datetime import datetime, timedelta

def load_large_theaters():
    large_theaters = {}
    with open('/content/drive/MyDrive/crawl/data/large_theaters.csv', 'r', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        for row in reader:
            large_theaters[row['name']] = int(row['seatscale'])
    return large_theaters

def get_performances(service_key, start_date, end_date, page=1, rows=100, genre='GGGA'):
    url = "http://www.kopis.or.kr/openApi/restful/pblprfr"
    params = {
        'service': service_key,
        'stdate': start_date,
        'eddate': end_date,
        'cpage': page,
        'rows': rows,
        'shcate': genre,
    }
    response = requests.get(url, params=params)
    if response.status_code == 200:
        return ET.fromstring(response.content)
    else:
        print(f"Failed to fetch performances. Status code: {response.status_code}")
        return None

def find_best_match(theater_name, large_theaters, threshold=80):
    best_match = None
    best_score = 0
    for large_theater in large_theaters:
        score = fuzz.ratio(theater_name.lower(), large_theater.lower())
        if score > best_score:
            best_score = score
            best_match = large_theater

    if best_score >= threshold:
        return best_match, best_score
    return None, best_score

def save_performances_to_csv(performances, file_path):
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    with open(file_path, 'w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=['name', 'theater', 'start_date', 'end_date', 'matched_theater', 'exact_match', 'similarity_score'])
        writer.writeheader()
        writer.writerows(performances)

def main():
    end_date = datetime.now().strftime('%Y%m%d')
    start_date = (datetime.now() - timedelta(days=365)).strftime('%Y%m%d')

    large_theaters = load_large_theaters()
    performances = []
    page = 1

    while len(performances) < 1000:
        result = get_performances(service_key, start_date, end_date, page)
        if result is None:
            break

        for item in result.findall('.//db'):
            performance = {
                'name': item.findtext('prfnm'),
                'theater': item.findtext('fcltynm'),
                'start_date': item.findtext('prfpdfrom'),
                'end_date': item.findtext('prfpdto'),
            }

            matched_theater, similarity_score = find_best_match(performance['theater'], large_theaters)
            performance['matched_theater'] = matched_theater if matched_theater else ''
            performance['exact_match'] = 'true' if matched_theater and similarity_score == 100 else 'false'
            performance['similarity_score'] = similarity_score

            if matched_theater:
                performances.append(performance)
                print(f"Added performance: {performance['name']} at {performance['theater']}")
                print(f"Matched with: {matched_theater} (Score: {similarity_score})")
                print("-" * 50)

            if len(performances) >= 1000:
                break

        page += 1

    save_performances_to_csv(performances, '/content/drive/MyDrive/crawl/data/matched_performances.csv')
    print(f"Total performances saved: {len(performances)}")

if __name__ == "__main__":
    main()

NameError: name 'requests' is not defined